# 02 — Baseline TF-IDF + cosine (M1.4)
Precision@3 sur le pool annoté figé.

In [ ]:
# Se placer à la racine du projet (VS Code, Jupyter ou Colab)
import os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Racine du projet :", ROOT)

In [ ]:
from cvmatch.config import get_settings
from cvmatch.data import load_jobs, job_texts
from cvmatch.ranking import TfidfRanker
from cvmatch.evaluation import load_cvs, load_labels, evaluate_ranker, summarize
from cvmatch.preprocessing.cleaning import clean_text
from cvmatch.preprocessing.pii import mask_pii
s = get_settings()
jobs = load_jobs(s.jobs_path)
cvs = load_cvs(s.eval_dir / "cvs")
tfidf = TfidfRanker(job_texts(jobs, True))

In [ ]:
cv_id = next(iter(cvs))
for r in tfidf.rank(clean_text(mask_pii(cvs[cv_id])), top_k=3):
    print(round(r.score, 3), jobs.iloc[r.index]["job_title"])

In [ ]:
from pathlib import Path
lab = s.eval_dir / "labels.csv"
if not lab.exists():
    print("⚠ labels.csv absent : annotez le pool (README, étape 5). Labels proxy = test technique seulement.")
    lab = s.eval_dir / "proxy_labels.csv"
labels = load_labels(lab)
df = evaluate_ranker(tfidf, jobs, cvs, labels, preprocess=lambda t: clean_text(mask_pii(t)))
print(summarize(df, "B0_tfidf")); df